# exp_reranker_bakeoff — is a stronger off-the-shelf reranker the lever for the 2022 gap? (path A go/no-go)

Every *tuning* lever is exhausted (5 reranking levers §11, hard gate, soft blend). The one untested
question: does a **stronger, modern, off-the-shelf reranker** beat our from-scratch BioLinkBERT ensemble
(TREC22 0.575)? The gap to SOTA is only 0.037, so a better core reranker could close it.

Reranks the 2022 pool (+ 2021 judged pool, develop) zero-shot with **BGE-reranker-large** (strong MS-MARCO
cross-encoder) and **monoT5-3B-MED** (we already have it), on the R = elig_first representation (2022 rewards
eligibility). Compare to our ensemble (2022 = 0.575) and clf_R (2021 judged-pool ~0.66).

Go/no-go: if a zero-shot reranker >= our ensemble on 2022, path A is live -> LoRA-adapt it and go full pipeline.
If both are well below (as monoT5-MED was in-domain, ~0.45 on 2021), a bigger core needs the domain adaptation
that failed 6x -> path A is high-risk and the architecture ceiling stands. Resumable/cached.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q sentence-transformers transformers accelerate sentencepiece pytrec_eval datasets tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET']='1'; os.environ['HF_HOME']='/content/hf_cache'
import numpy as np, torch, pytrec_eval
from tqdm.auto import tqdm
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval, truncated_doc_text
DATA_ROOT='/content/drive/MyDrive/ct_data23'; cfg=ExperimentConfig(data_root=DATA_ROOT, pool_tag='nqs')
device='cuda' if torch.cuda.is_available() else 'cpu'
CACHE=cfg.path('data'); os.makedirs(CACHE, exist_ok=True)

corpus_ids, corpus_fields = load_corpus(cfg); id2f = dict(zip(corpus_ids, corpus_fields))
sets = load_eval(cfg, ['trec21','trec22'])
pool = json.load(open(cfg.path('data/pool_nqs.json')))
DS = {'trec22': (list(pool['trec22']), lambda t: pool['trec22'][t]),
      'trec21': (list(sets['trec21']['rel_dict']), lambda t: list(sets['trec21']['rel_dict'][t]))}
t2t = {'trec22': sets['trec22']['topic2text'], 'trec21': sets['trec21']['topic2text']}
rel = {'trec22': sets['trec22']['rel_dict'], 'trec21': sets['trec21']['rel_dict']}
def ndcg(scores, ds):
    run={}; qr={}
    for t in DS[ds][0]:
        docs=[d for d in DS[ds][1](t) if (t,d) in scores]
        if not docs: continue
        run[t]={d:scores[(t,d)] for d in docs}; qr[t]={d:int(r) for d,r in rel[ds][t].items()}
    ev=pytrec_eval.RelevanceEvaluator(qr,{'ndcg_cut.10'}).evaluate(run)
    return float(np.mean([v['ndcg_cut_10'] for v in ev.values()]))
print('loaded | ensemble baseline TREC22=0.575, clf_R TREC21 judged-pool~0.66')

In [ ]:
# Generic resumable rerank: score every (topic, pool-doc) on R = elig_first, cache per (reranker, dataset).
def rerank(name, score_fn, tok_for_trunc, reserve_extra=8):
    for ds in ['trec21','trec22']:
        path=f'{CACHE}/rr_{name}_{ds}.jsonl'; done=set(); sc={}
        if os.path.exists(path):
            for l in open(path):
                r=json.loads(l); sc[(r['t'],r['d'])]=r['s']; done.add(r['t'])
        todo=[t for t in DS[ds][0] if t not in done and t in t2t[ds]]
        if todo:
            with open(path,'a') as f:
                for t in tqdm(todo, desc=f'{name} {ds}'):
                    docs=[d for d in DS[ds][1](t) if d in id2f]
                    if not docs: continue
                    reserve=reserve_extra+len(tok_for_trunc.encode(t2t[ds][t], add_special_tokens=False))
                    dstr=[truncated_doc_text(tok_for_trunc, id2f[d], cfg, reserve=reserve, max_length=512) for d in docs]
                    for d,s in zip(docs, score_fn(t2t[ds][t], dstr)): f.write(json.dumps({'t':t,'d':d,'s':float(s)})+'\n')
                    f.flush()
        yield ds, {(r['t'],r['d']):r['s'] for r in (json.loads(l) for l in open(path))}

results={}
# 1) BGE-reranker-large (cross-encoder relevance scorer)
from sentence_transformers import CrossEncoder
bge=CrossEncoder('BAAI/bge-reranker-large', max_length=512, device=device)
for ds, sc in rerank('bge_rerank', lambda q,ds_: bge.predict([[q,d] for d in ds_], batch_size=64), bge.tokenizer):
    results[('bge_rerank',ds)]=ndcg(sc, ds)
del bge; torch.cuda.empty_cache()
print('bge-reranker done')

In [ ]:
# 2) monoT5-3B-MED (true/false margin), same elig_first docs
from transformers import T5Tokenizer, T5ForConditionalGeneration
mt_tok=T5Tokenizer.from_pretrained('castorini/monot5-3b-med-msmarco')
mt=T5ForConditionalGeneration.from_pretrained('castorini/monot5-3b-med-msmarco', torch_dtype=torch.float16, device_map='auto').eval()
TRUE=mt_tok('true',add_special_tokens=False).input_ids[0]; FALSE=mt_tok('false',add_special_tokens=False).input_ids[0]
@torch.no_grad()
def mt_score(q, docs, batch=16):
    out=[]
    for i in range(0,len(docs),batch):
        prompts=[f'Query: {q} Document: {d} Relevant:' for d in docs[i:i+batch]]
        enc=mt_tok(prompts,return_tensors='pt',padding=True,truncation=True,max_length=512).to(mt.device)
        dec=torch.zeros((enc['input_ids'].shape[0],1),dtype=torch.long,device=mt.device)
        lp=torch.log_softmax(mt(**enc,decoder_input_ids=dec).logits[:,0,:].float(),-1)
        out.extend((lp[:,TRUE]-lp[:,FALSE]).cpu().tolist())
    return out
for ds, sc in rerank('monot5_med', mt_score, mt_tok, reserve_extra=12):
    results[('monot5_med',ds)]=ndcg(sc, ds)
del mt; torch.cuda.empty_cache()

print(f'\n{"reranker":18s} {"TREC21 (dev)":>12s} {"TREC22 (test)":>14s}')
print('-'*46)
print(f'{"our ensemble":18s} {"~0.66 (clf_R)":>12s} {"0.575":>14s}')
for name in ['bge_rerank','monot5_med']:
    print(f'{name:18s} {results.get((name,"trec21"),float("nan")):>12.4f} {results.get((name,"trec22"),float("nan")):>14.4f}')
print('\nGO if a zero-shot reranker >= 0.575 on TREC22 (and competitive on TREC21) -> path A live, LoRA-adapt it.')
print('NO-GO if both are well below -> a stronger core needs domain adaptation that has failed 6x; ceiling stands.')